In [ ]:
!pip install osmnx geopandas networkx shapely

In [ ]:
   !python build_data.py

In [ ]:
!zip -r data.zip data
from google.colab import files
files.download("data.zip")

In [ ]:
!ls -la
!ls -la data

In [1]:
!zip -r data.zip data
from google.colab import files
files.download("data.zip")

	zip warning: name not matched: data

zip error: Nothing to do! (try: zip -r data.zip . -i data)


FileNotFoundError: Cannot find file: data.zip

In [2]:
import os
# Check the contents of the current directory to see what was generated
print('Current working directory:', os.getcwd())
print('Contents:', os.listdir('.'))


Current working directory: /content
Contents: ['.config', 'sample_data']


In [3]:
!pip install -q "osmnx>=2.0" geopandas networkx shapely

import os, time
import numpy as np, pandas as pd, networkx as nx
import osmnx as ox, geopandas as gpd
from shapely.ops import unary_union
from google.colab import files

ox.settings.timeout = 300
ox.settings.log_console = False

BBOX = (77.90, 30.22, 78.15, 30.42)      # left, bottom, right, top (Dehradun area)
BANDS = [5, 10, 15]                       # minutes
SPEED_FACTOR = 0.8                        # emergency speed vs OSM free-flow; calibrate later
BUFFER_M = 100
ISO_SERVICES = ["Hospital", "Police station", "Fire station"]
os.makedirs("data", exist_ok=True)

TAGS = {
    "amenity": ["hospital", "clinic", "police", "fire_station", "pharmacy", "blood_bank"],
    "emergency": ["ambulance_station", "assembly_point"],
    "healthcare": ["clinic", "blood_bank", "centre"],
    "aeroway": ["helipad"],
}

def retry(fn, tries=4, wait=20):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            print(f"Attempt {i+1} failed: {e}")
            if i == tries - 1:
                raise
            time.sleep(wait)

def classify(r):
    a, e, h, w = r.get("amenity"), r.get("emergency"), r.get("healthcare"), r.get("aeroway")
    if a == "hospital": return "Hospital"
    if a == "police": return "Police station"
    if a == "fire_station": return "Fire station"
    if e == "ambulance_station": return "Ambulance station"
    if a == "blood_bank" or h == "blood_bank": return "Blood bank"
    if a == "pharmacy": return "Pharmacy"
    if a == "clinic" or h in ("clinic", "centre"): return "Clinic"
    if e == "assembly_point": return "Assembly point"
    if w == "helipad": return "Helipad"
    return None

# 1. Facilities -----------------------------------------------------------
print("Downloading facilities from OpenStreetMap...")
raw = retry(lambda: ox.features_from_bbox(bbox=BBOX, tags=TAGS)).reset_index()
raw["category"] = raw.apply(classify, axis=1)
raw = raw[raw["category"].notna()].to_crs(32644)
raw["geometry"] = raw.geometry.centroid
for col in ["name", "operator", "phone", "contact:phone", "opening_hours"]:
    if col not in raw:
        raw[col] = None
raw["phone"] = raw["phone"].fillna(raw["contact:phone"])
fac = raw[["name", "category", "operator", "phone", "opening_hours", "geometry"]].copy()
fac["name"] = fac["name"].fillna(fac["category"])
key = (fac["name"] + fac["category"] + (fac.geometry.x // 100).astype(int).astype(str)
       + (fac.geometry.y // 100).astype(int).astype(str))
fac = fac[~key.duplicated()]
fac.to_crs(4326).to_file("data/facilities.geojson", driver="GeoJSON")
print(fac["category"].value_counts())
print("Saved data/facilities.geojson")

# 2. Isochrones -----------------------------------------------------------
try:
    print("Downloading road network (this can take a few minutes)...")
    G = retry(lambda: ox.graph_from_bbox(bbox=BBOX, network_type="drive"))
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    for _, _, _, d in G.edges(keys=True, data=True):
        d["tt"] = d["travel_time"] / SPEED_FACTOR
    G = ox.project_graph(G)
    edges = ox.graph_to_gdfs(G, nodes=False).reset_index()
    fac_g = fac.to_crs(G.graph["crs"])

    rows = []
    for svc in ISO_SERVICES:
        f = fac_g[fac_g["category"] == svc]
        if f.empty:
            print("No", svc, "found, skipping")
            continue
        sources = set(ox.distance.nearest_nodes(G, f.geometry.x, f.geometry.y))
        times = nx.multi_source_dijkstra_path_length(G, sources, weight="tt", cutoff=max(BANDS) * 60)
        tu = edges["u"].map(times).fillna(np.inf)
        tv = edges["v"].map(times).fillna(np.inf)
        edge_t = pd.concat([tu, tv], axis=1).max(axis=1)
        for b in BANDS:
            sel = edges[edge_t <= b * 60]
            if not sel.empty:
                poly = unary_union(sel.geometry.buffer(BUFFER_M)).simplify(15)
                rows.append({"service": svc, "minutes": b, "geometry": poly})
        print("Isochrones done for", svc)

    gpd.GeoDataFrame(rows, crs=G.graph["crs"]).to_crs(4326).to_file("data/isochrones.geojson", driver="GeoJSON")
    print("Saved data/isochrones.geojson")
except Exception as e:
    print("Isochrone step failed, but facilities were saved:", e)

# 3. Zip and download -----------------------------------------------------
print(os.listdir("data"))
!zip -r -q data.zip data
files.download("data.zip")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 1.2 MB/s eta 0:00:00
category
Hospital          84
Clinic            58
Police station     8
Pharmacy           6
Blood bank         5
Helipad            3
Fire station       2
Name: count, dtype: int64
Saved data/facilities.geojson
Attempt 1 failed: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and the request queue is full.

## What to do

Retry your request using exponential backoff: wait a short interval, then retry.
If the request fails again, double the wait time and retry again.

## For heavy or automated use

This public endpoint is intended for browser-based and human-operated tools.
For automated or high-volume use, run a local Overpass API instance:

https://github.com/b1tw153/Overpass-API/blob/main/README.md

Isochrones done for Hospital
Isochrones done for Police station
Isochrones done for Fire station
Saved data/is

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>